# 3. AIND Ephys Spike Sorting

Each sorter is its own task, as each is its own step of the AIND pipeline. This notebook builds `AINDEPhysSpikeSortingKilosort4ScanConfig`, expands it with `GridScanGenerationTask`, and runs `AINDEPhysSpikeSortingKilosort4Task` for each coordinate. The task sorts every preprocessed recording, removes empty units and spikes beyond the recording, and saves the sorting to `spikesorted_<name>/`.

The other sorters have the same shape in their own packages: `AINDEPhysSpikeSortingSpykingCircus2ScanConfig` (ships with spikeinterface) and `AINDEPhysSpikeSortingLupinScanConfig` (needs torch). Kilosort4 needs obi-one's `kilosort` extra. A sorter's parameters are spread across blocks by theme; a few of Kilosort4's are scaled down here for the toy probe. Leaving `preprocessing_output_path` as `None` instead sorts a toy recording made and preprocessed on the fly, so the stage can also run on its own.

## Imports

In [1]:
from pathlib import Path

from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.tasks.aind_ephys._03_spikesorting.kilosort4.blocks import (
    Kilosort4DriftCorrection,
    Kilosort4Preprocessing,
    Kilosort4Runtime,
    Kilosort4Templates,
)
from obi_one.scientific.tasks.aind_ephys._03_spikesorting.kilosort4.config import (
    AINDEPhysSpikeSortingKilosort4ScanConfig,
)
from obi_one.scientific.tasks.aind_ephys.blocks import JobKwargs

## Build the scan config

`preprocessing_output_path` is notebook 02's coordinate output folder.

In [2]:
obi_output = (Path.cwd() / "../../../../../../obi-output").resolve()
preprocessing_output_path = obi_output / "02_aind_ephys_preprocessing" / "grid_scan" / "0"
assert preprocessing_output_path.exists(), "Run notebook 02 first."

scan_config = AINDEPhysSpikeSortingKilosort4ScanConfig(
    info=Info(
        campaign_name="Toy spike sorting",
        campaign_description="Toy spike sorting of the toy recording.",
    ),
    initialize=AINDEPhysSpikeSortingKilosort4ScanConfig.Initialize(
        preprocessing_output_path=preprocessing_output_path,
    ),
    job_kwargs=JobKwargs(n_jobs=1),
    preprocessing=Kilosort4Preprocessing(whitening_range=16),
    drift_correction=Kilosort4DriftCorrection(nblocks=0),
    templates=Kilosort4Templates(nearest_chans=8, nearest_templates=32),
    runtime=Kilosort4Runtime(torch_device="cpu"),
)

## Generate the grid scan and run the sorting task

In [3]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=obi_output / "03_aind_ephys_spikesorting" / "grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

run_tasks_for_generated_scan(grid_scan)

  0%|          | 0/4 [00:00<?, ?it/s]

 25%|██▌       | 1/4 [00:00<00:00,  4.31it/s]

 50%|█████     | 2/4 [00:00<00:00,  4.03it/s]

 75%|███████▌  | 3/4 [00:00<00:00,  4.24it/s]

100%|██████████| 4/4 [00:00<00:00,  4.51it/s]

100%|██████████| 4/4 [00:00<00:00,  4.38it/s]

... (10 outputs left out)

100%|██████████| 1/1 [00:00<00:00, 322.37it/s]

100%|██████████| 1/1 [00:00<00:00, 294.19it/s]

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/spikeinterface/core/basesorting.py:646: UserWarning: The recording registered to this sorting object will not be saved to disk. Reloading the sorting later will not include the recording
  warnings.warn(


[PosixPath('/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/03_aind_ephys_spikesorting/grid_scan/0')]

## Load the sorting and score it against the ground truth

In [4]:
import spikeinterface as si
import spikeinterface.comparison as sc

ground_truth = si.load(obi_output / "00_toy_example_recording" / "ground_truth_sorting")
coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
for sorting_folder in sorted(coord_dir.glob("spikesorted_*")):
    sorting = si.load(sorting_folder)
    print(sorting_folder.name, "->", sorting)
    comparison = sc.compare_sorter_to_ground_truth(ground_truth, sorting)
    print(comparison.get_performance(method="pooled_with_average"))

spikesorted_block0_acquisition-ElectricalSeriesRaw_recording1 -> NumpyFolder (NumpyFolderSorting): 8 units - 1 segments - 25.0kHz


accuracy                0.403652
recall                  0.454025
precision                0.44134
false_discovery_rate     0.05866
miss_rate               0.045975
dtype: object
